In [1]:
from pathlib import Path
import pandas as pd
import re

# Locate the project folder
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

facility_file = (
    project_root
    / "data"
    / "interim"
    / "active_restaurant_facilities.csv"
)

facilities = pd.read_csv(
    facility_file,
    encoding="utf-8-sig",
    low_memory=False
)

print("Facility rows loaded:", len(facilities))
print("Columns loaded:", len(facilities.columns))
display(facilities.head())

Facility rows loaded: 27438
Columns loaded: 13


,facility_id,facility_name,facility_address,facility_city,facility_state,facility_zip,latest_inspection_date,program_count,owner_count,average_latest_score,minimum_latest_score,maximum_latest_score,special_venue
0,FA0001114,CHRIS & PITTS BBQ #6,9243 LAKEWOOD BLVD,DOWNEY,CA,90240,2024-11-13,1,1,90.0,90,90,False
1,FA0001155,FRATERNAL ORDER OF EAGLES,13018 W WASHINGTON BLVD,LOS ANGELES,CA,90066,2026-06-18,1,1,97.0,97,97,False
2,FA0001334,HARBOR ROOM BAR,195 CULVER BLVD,PLAYA DEL REY,CA,90293,2024-07-25,1,1,98.0,98,98,False
3,FA0001348,JACK IN THE BOX,9433 RESEDA BLVD,NORTHRIDGE,CA,91324,2025-04-23,1,1,98.0,98,98,False
4,FA0001371,AVALON SEAFOOD,20 GREEN PLEASURE PIER,AVALON,CA,90704,2025-07-14,1,1,93.0,93,93,False


In [2]:
# Normalize restaurant names for keyword matching

facilities["name_normalized"] = (
    facilities["facility_name"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Strong indicators of Chinese or Greater Chinese cuisine
strong_keywords = [
    "CHINESE",
    "CHINA",
    "SZECHUAN",
    "SICHUAN",
    "HUNAN",
    "CANTON",
    "CANTONESE",
    "MANDARIN",
    "SHANGHAI",
    "SHANGHAINESE",
    "BEIJING",
    "PEKING",
    "DIM SUM",
    "HOT POT",
    "HOTPOT",
    "TAIWAN",
    "TAIWANESE",
    "HONG KONG",
    "HK CAFE",
    "XIAN",
    "XI AN",
    "CHONGQING",
    "GUANGDONG",
    "YUNNAN",
    "PANDA EXPRESS",
    "LANZHOU",
    "XINJianG"
    "PANDA",
    "CHINATOWN EXPRESS",
    "CHINATOWN KITCHEN",
    "SHANGHAILANDER",
    "SHAXIAN"
]

# Possible indicators requiring manual review
review_keywords = [
    "WOK",
    "DUMPLINGS",
    "WOKS",
    "XIANG",
    "XIANGJI",
    "XIANGYU",
    "LUYIXIAN",
    "QIANLIXIANG",
    "WOKANO",
    "WOKCANO",
    "TIGAWOK",
    "ROBOWOK",
    "WOKSHOP",
    "DRAGON",
    "DUMPLING",
    "NOODLE",
    "NOODLES",
    "FORTUNE"
]

def build_keyword_pattern(keywords):
    escaped_keywords = [
        re.escape(keyword)
        for keyword in keywords
    ]

    return (
        r"(?<![A-Z0-9])(?:"
        + "|".join(escaped_keywords)
        + r")(?![A-Z0-9])"
    )

strong_pattern = build_keyword_pattern(strong_keywords)
review_pattern = build_keyword_pattern(review_keywords)

facilities["chinese_strong_match"] = (
    facilities["name_normalized"]
    .str.contains(strong_pattern, regex=True, na=False)
)

facilities["chinese_review_match"] = (
    facilities["name_normalized"]
    .str.contains(review_pattern, regex=True, na=False)
    & ~facilities["chinese_strong_match"]
)

print(
    "Strong keyword candidates:",
    facilities["chinese_strong_match"].sum()
)

print(
    "Additional review candidates:",
    facilities["chinese_review_match"].sum()
)

Strong keyword candidates: 559
Additional review candidates: 294


In [3]:
strong_candidates = facilities.loc[
    facilities["chinese_strong_match"],
    [
        "facility_id",
        "facility_name",
        "facility_city",
        "facility_address",
        "special_venue"
    ]
].sort_values(["facility_city", "facility_name"])

review_candidates = facilities.loc[
    facilities["chinese_review_match"],
    [
        "facility_id",
        "facility_name",
        "facility_city",
        "facility_address",
        "special_venue"
    ]
].sort_values(["facility_city", "facility_name"])

display(strong_candidates.head(30))
display(review_candidates.head(30))

,facility_id,facility_name,facility_city,facility_address,special_venue
4423,FA0043093,PANDA EXPRESS #792,ACTON,2919 LOS FELIZ BLVD STE #4,False
5867,FA0054366,WENCES ACTON CHINESE,ACTON,3620 SMITH AVE # L,False
3898,FA0038423,MANDARIN LOTUS FINE CHINESE,AGOURA HILLS,5015 KANAN RD,False
7295,FA0068247,PANDA EXPRESS,AGOURA HILLS,29145 CANWOOD ST STE A-5,False
176,FA0005102,ACC CHINESE FAST FOOD,ALHAMBRA,38 S PALM AVE,False
461,FA0008150,BAMBOO GARDEN CHINESE REST,ALHAMBRA,2632 W VALLEY BLVD,False
22563,FA0332002,DIM SUM DUMPLING HOUSE,ALHAMBRA,417 W MAIN ST,False
7060,FA0066116,DIM SUM HOUSE,ALHAMBRA,500 W MAIN ST ABC,False
19603,FA0318430,FAMOUS CHINESE SNACKS,ALHAMBRA,345 E MAIN ST 103,False
24026,FA0354057,KUES CHINESE RESTAURANT & BAR,ALHAMBRA,201 W MAIN ST,False


,facility_id,facility_name,facility_city,facility_address,special_venue
19099,FA0312973,101 NOODLE EXPRESS,ALHAMBRA,1408 E VALLEY BLVD,False
13228,FA0261929,KOSUKE NOODLE,ALHAMBRA,618 W MAIN ST STE B,False
18358,FA0308389,LUYIXIAN,ALHAMBRA,2 E VALLEY BLVD STE 1E,False
21625,FA0329065,MEOW RICE NOODLE,ALHAMBRA,555 W MAIN ST UNIT A,False
2476,FA0026271,NEW NOODLE CITY,ALHAMBRA,628 W VALLEY BLVD,False
25114,FA0357717,NOODLE WORLD,ALHAMBRA,700 W VALLEY BLVD,False
11584,FA0242991,SAM'S NOODLE STATION,ALHAMBRA,1281 E VALLEY BLVD,False
24738,FA0356588,STUDENT DAYS RICE NOODLES BBQ,ALHAMBRA,640 W VALLEY BLVD,False
14879,FA0276240,XIANG LA HUI,ALHAMBRA,621 W MAIN ST,False
24149,FA0354383,XIANG LA LOU,ALHAMBRA,500 W VALLEY BLVD,False


In [4]:
# Count facilities matched by each keyword

keyword_results = []

for category, keywords in [
    ("strong", strong_keywords),
    ("review", review_keywords)
]:
    for keyword in keywords:
        match_count = (
            facilities["name_normalized"]
            .str.contains(
                re.escape(keyword),
                regex=True,
                na=False
            )
            .sum()
        )

        keyword_results.append(
            {
                "category": category,
                "keyword": keyword,
                "facility_count": match_count
            }
        )

keyword_summary = (
    pd.DataFrame(keyword_results)
    .sort_values(
        ["category", "facility_count"],
        ascending=[True, False]
    )
)

display(
    keyword_summary.loc[
        keyword_summary["facility_count"] > 0
    ]
)

,category,keyword,facility_count
46,review,NOODLE,134
31,review,WOK,99
45,review,DUMPLING,54
44,review,DRAGON,34
47,review,NOODLES,23
34,review,XIANG,11
32,review,DUMPLINGS,8
48,review,FORTUNE,8
40,review,WOKCANO,4
33,review,WOKS,3


In [5]:
# Identify review candidates with signals of other cuisines

other_cuisine_keywords = [
    "RAMEN",
    "SUSHI",
    "JAPANESE",
    "UDON",
    "SOBA",
    "PHO",
    "VIETNAMESE",
    "THAI",
    "MYUNG IN",
    "MYUNGIN",
    "KOREAN",
    "TERIYAKI"
]

other_cuisine_pattern = build_keyword_pattern(
    other_cuisine_keywords
)

facilities["other_cuisine_signal"] = (
    facilities["name_normalized"]
    .str.contains(
        other_cuisine_pattern,
        regex=True,
        na=False
    )
)

# Create transparent candidate categories
facilities["chinese_candidate_status"] = "not_identified"

facilities.loc[
    facilities["chinese_review_match"],
    "chinese_candidate_status"
] = "review_candidate"

facilities.loc[
    facilities["chinese_review_match"]
    & facilities["other_cuisine_signal"],
    "chinese_candidate_status"
] = "review_likely_other_cuisine"

facilities.loc[
    facilities["chinese_strong_match"],
    "chinese_candidate_status"
] = "strong_candidate"

display(
    facilities["chinese_candidate_status"]
      .value_counts()
      .rename_axis("candidate_status")
      .reset_index(name="facility_count")
)

,candidate_status,facility_count
0,not_identified,26585
1,strong_candidate,559
2,review_candidate,270
3,review_likely_other_cuisine,24


In [6]:
likely_other_cuisine = facilities.loc[
    facilities["chinese_candidate_status"]
    == "review_likely_other_cuisine",
    [
        "facility_id",
        "facility_name",
        "facility_city",
        "facility_address"
    ]
].sort_values(["facility_city", "facility_name"])

print(
    "Review candidates with other-cuisine signals:",
    len(likely_other_cuisine)
)

display(likely_other_cuisine.head(30))

Review candidates with other-cuisine signals: 24


,facility_id,facility_name,facility_city,facility_address
18203,FA0307681,THAI NOODLE KING,BELLFLOWER,9887 ALONDRA BLVD
27172,FA0400810,WESTERN & THAI WOK,BURBANK,1311 N HOLLYWOOD WAY
5999,FA0055385,WOKCANO JAPANESE KITCHEN,BURBANK,150 S SAN FERNANDO BLVD
21145,FA0326993,ALL ABOUT PHO NOODLE BAR,CERRITOS,11265 183RD ST
24496,FA0355891,AN NOODLES AND PHO,LA MIRADA,15534 LA MIRADA BLVD
16718,FA0296526,AFURI RAMEN + DUMPLING,LOS ANGELES,688 MATEO ST
448,FA0007952,CASA NOODLE TERIYAKI,LOS ANGELES,5930 S MAIN ST STE #103
21169,FA0327066,KINARI ABURI SUSHI & NOODLE,LOS ANGELES,801 N FAIRFAX AVE
20990,FA0326434,MAE MALAI THAI HOUSE OF NOODLES,LOS ANGELES,5445 HOLLYWOOD BLVD
10667,FA0221973,MYUNG IN DUMPLINGS,LOS ANGELES,3109 W OLYMPIC BLVD STE B


In [7]:
# Record the exact keywords matched by each facility

def find_matched_keywords(name, keywords):
    if pd.isna(name):
        return ""

    matches = []

    for keyword in keywords:
        keyword_pattern = (
            r"(?<![A-Z0-9])"
            + re.escape(keyword)
            + r"(?![A-Z0-9])"
        )

        if re.search(keyword_pattern, name):
            matches.append(keyword)

    return ", ".join(matches)

facilities["matched_strong_keywords"] = (
    facilities["name_normalized"]
    .apply(
        lambda name: find_matched_keywords(
            name,
            strong_keywords
        )
    )
)

facilities["matched_review_keywords"] = (
    facilities["name_normalized"]
    .apply(
        lambda name: find_matched_keywords(
            name,
            review_keywords
        )
    )
)

facilities["matched_other_cuisine_keywords"] = (
    facilities["name_normalized"]
    .apply(
        lambda name: find_matched_keywords(
            name,
            other_cuisine_keywords
        )
    )
)

In [8]:
# Export all automatically identified candidates for review

candidate_facilities = facilities.loc[
    facilities["chinese_candidate_status"]
    != "not_identified"
].copy()

candidate_columns = [
    "facility_id",
    "facility_name",
    "name_normalized",
    "facility_address",
    "facility_city",
    "facility_state",
    "facility_zip",
    "chinese_candidate_status",
    "matched_strong_keywords",
    "matched_review_keywords",
    "matched_other_cuisine_keywords",
    "special_venue",
    "program_count",
    "average_latest_score"
]

candidate_file = (
    project_root
    / "data"
    / "interim"
    / "chinese_restaurant_candidates.csv"
)

candidate_facilities[candidate_columns].to_csv(
    candidate_file,
    index=False,
    encoding="utf-8-sig"
)

print("Candidate file saved to:")
print(candidate_file)
print("Candidate rows saved:", len(candidate_facilities))

display(
    candidate_facilities["chinese_candidate_status"]
      .value_counts()
      .rename_axis("candidate_status")
      .reset_index(name="facility_count")
)

Candidate file saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\chinese_restaurant_candidates.csv
Candidate rows saved: 853


,candidate_status,facility_count
0,strong_candidate,559
1,review_candidate,270
2,review_likely_other_cuisine,24


In [9]:
# Count keyword matches among unresolved review candidates

unresolved_review = facilities.loc[
    facilities["chinese_candidate_status"]
    == "review_candidate"
].copy()

unresolved_keyword_results = []

for keyword in review_keywords:
    count = (
        unresolved_review["name_normalized"]
        .str.contains(
            re.escape(keyword),
            regex=True,
            na=False
        )
        .sum()
    )

    unresolved_keyword_results.append(
        {
            "keyword": keyword,
            "unresolved_facility_count": count
        }
    )

unresolved_keyword_summary = (
    pd.DataFrame(unresolved_keyword_results)
    .sort_values(
        "unresolved_facility_count",
        ascending=False
    )
)

display(unresolved_keyword_summary)

,keyword,unresolved_facility_count
15,NOODLE,110
0,WOK,73
14,DUMPLING,48
13,DRAGON,26
16,NOODLES,19
3,XIANG,10
1,DUMPLINGS,6
17,FORTUNE,6
9,WOKCANO,3
2,WOKS,3


In [10]:
panda_review = unresolved_review.loc[
    unresolved_review["name_normalized"]
    .str.contains(r"\bPANDA\b", regex=True, na=False),
    [
        "facility_id",
        "facility_name",
        "facility_city",
        "facility_address",
        "matched_review_keywords"
    ]
].sort_values(["facility_name", "facility_city"])

print("Unresolved PANDA candidates:", len(panda_review))
display(panda_review)

Unresolved PANDA candidates: 3


,facility_id,facility_name,facility_city,facility_address,matched_review_keywords
9362,FA0164981,PANDA WOK,COVINA,581 N AZUSA AVE,WOK
17835,FA0304920,PANDA WOK,NORTH HOLLYWOOD,8025 WEBB AVE,WOK
4517,FA0043749,PANDA WOK,SYLMAR,13205 GLADSTONE AVE,WOK


In [11]:
# Compare old substring matching with new complete-word matching

all_candidate_keywords = (
    strong_keywords + review_keywords
)

substring_pattern = "|".join(
    re.escape(keyword)
    for keyword in all_candidate_keywords
)

complete_word_pattern = build_keyword_pattern(
    all_candidate_keywords
)

old_substring_match = (
    facilities["name_normalized"]
    .str.contains(
        substring_pattern,
        regex=True,
        na=False
    )
)

new_complete_word_match = (
    facilities["name_normalized"]
    .str.contains(
        complete_word_pattern,
        regex=True,
        na=False
    )
)

removed_by_word_boundary = facilities.loc[
    old_substring_match & ~new_complete_word_match,
    [
        "facility_id",
        "facility_name",
        "facility_city",
        "facility_address"
    ]
].sort_values(["facility_name", "facility_city"])

print(
    "Records removed by complete-word matching:",
    len(removed_by_word_boundary)
)

display(removed_by_word_boundary.head(70))

Records removed by complete-word matching: 5


,facility_id,facility_name,facility_city,facility_address
20251,FA0321970,DRAGONFLY THAI BISTRO,ROLLING HILLS ESTATES,50 PENINSULA CTR
22630,FA0332198,EL CHINANTECO MEXICAN GRILL,LOS ANGELES,800 S MAIN ST
27110,FA0400375,LAS CHINAMPAS RESTAURANT INC,WHITTIER,8025 S PIONEER BLVD
13230,FA0261949,RESTAURANTE Y PUPUSERIA LAS CHINAMAS,MAYWOOD,4535 SLAUSON AVE
8725,FA0155839,SCOOPS CHINATOWN,LOS ANGELES,727 N BROADWAY WAY STE 125


In [12]:
# Create the high-confidence Chinese restaurant seed dataset

high_confidence_chinese = facilities.loc[
    facilities["chinese_candidate_status"]
    == "strong_candidate"
].copy()

# Exclude unusual large venues from the standard competitor seed
high_confidence_standard = high_confidence_chinese.loc[
    ~high_confidence_chinese["special_venue"]
].copy()

print(
    "High-confidence Chinese candidates:",
    len(high_confidence_chinese)
)

print(
    "High-confidence standard facilities:",
    len(high_confidence_standard)
)

print(
    "High-confidence special venues:",
    high_confidence_chinese["special_venue"].sum()
)

city_summary = (
    high_confidence_standard
    .groupby("facility_city")
    .agg(
        chinese_candidate_count=("facility_id", "nunique"),
        average_score=("average_latest_score", "mean")
    )
    .sort_values(
        "chinese_candidate_count",
        ascending=False
    )
    .reset_index()
)

city_summary["average_score"] = (
    city_summary["average_score"].round(2)
)

display(city_summary.head(20))

High-confidence Chinese candidates: 559
High-confidence standard facilities: 559
High-confidence special venues: 0


,facility_city,chinese_candidate_count,average_score
0,LOS ANGELES,136,91.90
1,MONTEREY PARK,15,90.07
2,INDUSTRY,13,87.23
3,TORRANCE,13,86.23
4,ALHAMBRA,12,88.00
5,POMONA,11,92.36
6,SAN GABRIEL,11,86.27
7,ARCADIA,11,92.64
8,ROSEMEAD,10,89.50
9,LANCASTER,9,95.78


In [13]:
# Compare Chinese candidate counts with all restaurants by city

standard_facilities = facilities.loc[
    ~facilities["special_venue"]
].copy()

all_restaurants_by_city = (
    standard_facilities
    .groupby("facility_city")
    .agg(
        all_restaurant_count=("facility_id", "nunique")
    )
    .reset_index()
)

city_market_summary = (
    all_restaurants_by_city
    .merge(
        city_summary,
        on="facility_city",
        how="left"
    )
)

city_market_summary["chinese_candidate_count"] = (
    city_market_summary["chinese_candidate_count"]
    .fillna(0)
    .astype(int)
)

city_market_summary["chinese_candidate_share_pct"] = (
    city_market_summary["chinese_candidate_count"]
    / city_market_summary["all_restaurant_count"]
    * 100
).round(2)

# Cities with the largest number of high-confidence candidates
display(
    city_market_summary
    .sort_values(
        "chinese_candidate_count",
        ascending=False
    )
    .head(20)
)

# Cities with the highest candidate share
# Require at least 20 restaurants to avoid tiny denominators
display(
    city_market_summary.loc[
        city_market_summary["all_restaurant_count"] >= 20
    ]
    .sort_values(
        "chinese_candidate_share_pct",
        ascending=False
    )
    .head(20)
)

,facility_city,all_restaurant_count,chinese_candidate_count,average_score,chinese_candidate_share_pct
78,LOS ANGELES,7265,136,91.90,1.87
87,MONTEREY PARK,217,15,90.07,6.91
140,TORRANCE,656,13,86.23,1.98
56,INDUSTRY,151,13,87.23,8.61
3,ALHAMBRA,267,12,88.00,4.49
5,ARCADIA,283,11,92.64,3.89
118,SAN GABRIEL,247,11,86.27,4.45
106,POMONA,324,11,92.36,3.40
114,ROSEMEAD,177,10,89.50,5.65
40,EL MONTE,250,9,89.89,3.60


,facility_city,all_restaurant_count,chinese_candidate_count,average_score,chinese_candidate_share_pct
56,INDUSTRY,151,13,87.23,8.61
87,MONTEREY PARK,217,15,90.07,6.91
49,HACIENDA HEIGHTS,101,6,93.33,5.94
33,CUDAHY,34,2,90.00,5.88
114,ROSEMEAD,177,10,89.50,5.65
30,COMMERCE,90,5,87.80,5.56
35,DIAMOND BAR,133,7,94.14,5.26
134,SUNLAND,41,2,97.50,4.88
127,SIGNAL HILL,42,2,87.50,4.76
3,ALHAMBRA,267,12,88.00,4.49


In [14]:
high_confidence_file = (
    project_root
    / "data"
    / "interim"
    / "high_confidence_chinese_candidates.csv"
)

high_confidence_standard.to_csv(
    high_confidence_file,
    index=False,
    encoding="utf-8-sig"
)

print("Rows saved:", len(high_confidence_standard))
print("File saved to:")
print(high_confidence_file)

Rows saved: 559
File saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\high_confidence_chinese_candidates.csv


In [15]:
# Flag cities with enough observations for a more stable comparison

city_market_summary["stable_comparison"] = (
    (city_market_summary["all_restaurant_count"] >= 100)
    & (city_market_summary["chinese_candidate_count"] >= 5)
)

stable_city_comparison = (
    city_market_summary.loc[
        city_market_summary["stable_comparison"]
    ]
    .sort_values(
        "chinese_candidate_share_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(stable_city_comparison.head(20))

,facility_city,all_restaurant_count,chinese_candidate_count,average_score,chinese_candidate_share_pct,stable_comparison
0,INDUSTRY,151,13,87.23,8.61,True
1,MONTEREY PARK,217,15,90.07,6.91,True
2,HACIENDA HEIGHTS,101,6,93.33,5.94,True
3,ROSEMEAD,177,10,89.50,5.65,True
4,DIAMOND BAR,133,7,94.14,5.26,True
5,ALHAMBRA,267,12,88.00,4.49,True
6,SAN GABRIEL,247,11,86.27,4.45,True
7,ROWLAND HEIGHTS,220,9,88.22,4.09,True
8,ARCADIA,283,11,92.64,3.89,True
9,EL MONTE,250,9,89.89,3.60,True


In [16]:
city_summary_file = (
    project_root
    / "data"
    / "interim"
    / "preliminary_city_competition_summary.csv"
)

city_market_summary.to_csv(
    city_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("Cities saved:", len(city_market_summary))
print("Stable-comparison cities:", len(stable_city_comparison))
print("File saved to:")
print(city_summary_file)

Cities saved: 160
Stable-comparison cities: 33
File saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\preliminary_city_competition_summary.csv
